# Knee MRI: v5 label experiment (fold 0, 160 px, EfficientNet-B0)

Trains fold 0 on three label sets with the **baseline recipe** (12 epochs, `--train_slices 16`, `--bs 4 --accum 2`, default seed 42)
and scores all of them on the **fixed old labels** (`--eval_labels` = baseline `oof_all.csv`), so the numbers are comparable with
the baseline (macro ~0.779, KEY-4 ~0.755).

| arm | training labels |
|---|---|
| `v4mix` | existing `labels_soft_v4.csv` (calibrated, A+B mix) |
| `v5b` | calibrated, **B only** (`--b_only`) |
| `v5c` | calibrated, B for 9 labels, A+B mix for MCL / Lateral Meniscus / Fracture (`--a_labels`) |

**Attach as inputs:** the competition data, the `knee-mri-pipeline` dataset **updated with `knee_pipeline_v5.zip`** (needs `knee_pipeline/` at top level),
the label notebook output (`A.csv`, `B.csv`, `labels_soft_v4.csv`), the 160 px cache notebook output, the baseline training notebook output (`oof_all.csv`),
optionally the seed-7/11 notebook output. **Settings:** GPU T4 x2, Internet ON (`--pretrained`). Run with **Save & Run All**.
Cells 4 (1-epoch smoke test) stops the notebook before the long jobs if anything is broken. `train.py` v5 was never run before this notebook.

In [1]:
# ---- 1. paths and discovery (prints every candidate; override below if it picks the wrong one) ----
import os, sys, glob, json, shutil, subprocess, time, threading
import numpy as np, pandas as pd
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langdetect"], check=False)

IN, W = "/kaggle/input", "/kaggle/working"
CODE_OVERRIDE = DATA_OVERRIDE = LAB_OVERRIDE = CACHE_OVERRIDE = BASE_OVERRIDE = None   # set to a path to force

def find_files(name, max_depth=6):
    """os.walk limited to max_depth below /kaggle/input (a recursive glob would crawl the DICOM tree)."""
    hits, base = [], IN.rstrip("/").count("/")
    for root, dirs, files in os.walk(IN):
        if root.count("/") - base >= max_depth:
            dirs[:] = []
        if name in files:
            hits.append(os.path.join(root, name))
    return sorted(hits)

def pick(label, hits, override=None, ok=lambda p: True):
    if override:
        print(f"{label}: OVERRIDE {override}"); return override
    good = [h for h in hits if ok(h)]
    print(f"{label}: {len(hits)} found, {len(good)} usable")
    for h in hits: print("   ", "*" if h in good else " ", h)
    assert good, f"no usable {label}; set the override in this cell"
    return good[0]

has = lambda path, s: s in open(path, errors="ignore").read()
CODE = os.path.dirname(pick("train.py (v5)", [p for p in find_files("train.py") if p.endswith("knee_pipeline/train.py")],
                            CODE_OVERRIDE and CODE_OVERRIDE + "/train.py",
                            ok=lambda p: has(p, "eval_labels") and os.path.exists(os.path.dirname(p) + "/labeler_combine.py")
                                         and has(os.path.dirname(p) + "/labeler_combine.py", "a_labels")))
DATA = os.path.dirname(pick("competition train.csv", [p for p in find_files("train.csv") if "/notebooks/" not in p],
                            DATA_OVERRIDE and DATA_OVERRIDE + "/train.csv"))
lab_dirs = [os.path.dirname(p) for p in find_files("labels_soft_v4.csv")]
LAB = pick("label dir (A.csv + B.csv + labels_soft_v4.csv)", lab_dirs, LAB_OVERRIDE,
           ok=lambda d: os.path.exists(d + "/A.csv") and os.path.exists(d + "/B.csv"))
BASE = pick("baseline oof_all.csv", find_files("oof_all.csv"), BASE_OVERRIDE)
SEED_OOF = {n: next(iter([p for p in find_files("oof.csv") if f"/exp_{n}/" in p]), None) for n in ("seed7", "seed11")}
print("seed oofs:", SEED_OOF)

# cache dir: must be the 160 px one (last dim 160), not the 224 px cache
cands = []
for p in find_files("cache_index.csv"):
    d = os.path.dirname(p)
    npys = [f for f in os.listdir(d) if f.endswith(".npy")]
    shape = np.load(f"{d}/{npys[0]}", mmap_mode="r").shape if npys else None
    print("   cache candidate", d, "| npy files:", len(npys), "| shape:", shape)
    if shape and shape[-1] == 160 and len(npys) >= 4400:
        cands.append(d)
CACHE = CACHE_OVERRIDE or (cands[0] if cands else None)
assert CACHE, "no 160 px cache with >=4400 files found; set CACHE_OVERRIDE"
print("\nCODE ", CODE, "\nDATA ", DATA, "\nLAB  ", LAB, "\nBASE ", BASE, "\nCACHE", CACHE)

import torch
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "no GPU"
assert len(pd.read_csv(f"{LAB}/B.csv")) == 4407 and len(pd.read_csv(f"{LAB}/A.csv")) == 4407, "A.csv / B.csv should have 4,407 rows"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.1 MB/s eta 0:00:00
train.py (v5): 1 found, 1 usable
    * /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py
competition train.csv: 1 found, 1 usable
    * /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv
label dir (A.csv + B.csv + labels_soft_v4.csv): 1 found, 1 usable
    * /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-label
baseline oof_all.csv: 1 found, 1 usable
    * /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv
seed oofs: {'seed7': '/kaggle/input/notebooks/theokorir/knee-mri-seed7-11-cache224-train/exp_seed7/oof.csv', 'seed11': '/kaggle/input/notebooks/theokorir/knee-mri-seed7-11-cache224-train/exp_seed11/oof.csv'}
   cache candidate /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache | npy files: 4407 | shape: (3, 24, 160, 160)

CODE  /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline 
DATA  /kaggle/input/competi

In [2]:
# ---- 2. build the three label sets (CPU, about a minute) ----
os.makedirs(f"{W}/labels", exist_ok=True)
env = dict(os.environ, KNEE_DATA_DIR=DATA, PYTHONPATH=CODE)
ARMS = {"v4mix": f"{LAB}/labels_soft_v4.csv",
        "v5b":   f"{W}/labels/labels_soft_v5b.csv",
        "v5c":   f"{W}/labels/labels_soft_v5c.csv"}
A_LABELS = "MCL,Lateral Meniscus,Fracture"

def combine(out, *extra):
    cmd = [sys.executable, "labeler_combine.py", "combine", "--train_csv", f"{DATA}/train.csv",
           "--a", f"{LAB}/A.csv", "--b", f"{LAB}/B.csv", "--out", out, "--calibrate_b", *extra]
    r = subprocess.run(cmd, env=env, cwd=CODE, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-3000:]); raise RuntimeError("combine failed")

combine(ARMS["v5b"], "--b_only")
combine(ARMS["v5c"], "--a_labels", A_LABELS)

# sanity: same studies, same order, and how far each arm is from the old labels
LABELS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
base = pd.read_csv(BASE)
arms = {k: pd.read_csv(v) for k, v in ARMS.items()}
for k, d in arms.items():
    assert d["StudyInstanceUID"].tolist() == arms["v4mix"]["StudyInstanceUID"].tolist(), f"{k}: study order differs"
    assert not d[[f"y_{l}" for l in LABELS]].isna().any().any(), f"{k}: NaN labels"
old = base.set_index("StudyInstanceUID")
rows = []
for l in LABELS:
    r = {"label": l}
    for k, d in arms.items():
        y = d.set_index("StudyInstanceUID")[f"y_{l}"]
        r[f"{k}_mean"] = round(y.mean(), 3)
        r[f"{k}_vs_old_spearman"] = round(y.corr(old[f"y_{l}"].reindex(y.index), method="spearman"), 3)
    rows.append(r)
print(pd.DataFrame(rows).set_index("label").to_string())
print("\nold-label mean (baseline y_):", round(float(np.mean([old[f"y_{l}"].mean() for l in LABELS])), 3))

there. Languages: {'tr': 546, 'hr': 406, 'el': 321, 'bg': 220, 'nl': 153}
                  n_gold  n_pos  n_neg  auc_A  auc_B  auc_mean  kappa_A_vs_B_all  prev_A  prev_B
label                                                                                           
ACL                   58     24     34  0.844  0.956     0.954             0.635   0.086   0.149
MCL                   58      9     49  0.811  0.929     0.943             0.477   0.049   0.129
Medial Meniscus       58     26     32  0.704  0.924     0.886             0.452   0.184   0.411
Lateral Meniscus      58     23     35  0.778  0.822     0.851             0.477   0.123   0.165
Medial OA             58     15     43  0.610  0.881     0.871             0.290   0.124   0.334
Lateral OA            58     11     47  0.538  0.864     0.816             0.379   0.099   0.165
PF OA                 58     21     37  0.645  0.805     0.796             0.436   0.195   0.366
Effusion              58     35     23  0.625  0.674 

In [3]:
# ---- 3. job definitions (edit QUEUES to change what runs where) ----
QUEUES = {0: ["v5b", "v5c"], 1: ["v4mix"]}          # jobs on one GPU run one after another
COMMON = ["--only_folds", "0", "--epochs", "12", "--pretrained", "--workers", "2", "--train_slices", "16",
          "--bs", "4", "--accum", "2"]               # baseline recipe; --seed left at its default (42)

def train_cmd(name, out, labels_path, extra=()):
    return [sys.executable, f"{CODE}/train.py", "--cache_dir", CACHE, "--labels", labels_path,
            "--eval_labels", BASE, "--out", out, *COMMON, *extra]
print(train_cmd("v5b", f"{W}/exp_v5b", ARMS["v5b"]))

['/usr/bin/python3', '/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py', '--cache_dir', '/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache', '--labels', '/kaggle/working/labels/labels_soft_v5b.csv', '--eval_labels', '/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv', '--out', '/kaggle/working/exp_v5b', '--only_folds', '0', '--epochs', '12', '--pretrained', '--workers', '2', '--train_slices', '16', '--bs', '4', '--accum', '2']


In [4]:
# ---- 4. SMOKE TEST: 1 epoch on one GPU with the v5b labels. Stops the notebook before the long jobs if anything is wrong. ----
SMOKE = True
if SMOKE:
    out = f"{W}/smoke"; shutil.rmtree(out, ignore_errors=True)
    cmd = train_cmd("smoke", out, ARMS["v5b"])
    cmd[cmd.index("--epochs") + 1] = "1"
    cmd += ["--bootstrap", "20"]
    t0 = time.time()
    r = subprocess.run(cmd, env=dict(env, CUDA_VISIBLE_DEVICES="0"), cwd=CODE, capture_output=True, text=True, timeout=2400)
    print("\n".join(r.stdout.splitlines()[-25:]))
    if r.returncode:
        print(r.stderr[-4000:]); raise RuntimeError("smoke test failed; fix before running the long jobs")
    o = pd.read_csv(f"{out}/oof.csv"); rep = json.load(open(f"{out}/cv_report.json"))
    done = o["p_ACL"].notna()
    assert done.sum() > 500, f"only {done.sum()} fold-0 studies scored"
    # the whole point of --eval_labels: y_ in oof.csv must be the OLD labels, ytrain_ the NEW training targets
    m = o.merge(base[["StudyInstanceUID"] + [f"y_{l}" for l in LABELS]], on="StudyInstanceUID", suffixes=("", "_base"))
    for l in LABELS:
        assert np.allclose(m[f"y_{l}"], m[f"y_{l}_base"]), f"y_{l} in oof.csv is not the fixed eval label"
    assert not np.allclose(m["ytrain_ACL"], m["y_ACL"]), "ytrain_ equals y_: training labels were not the new ones"
    assert rep["eval_labels"] == BASE and (o.fold == base.set_index("StudyInstanceUID").fold.reindex(o.StudyInstanceUID).values).all()
    print(f"\nSMOKE OK in {(time.time()-t0)/60:.1f} min | fold-0 studies scored: {int(done.sum())} | 1-epoch macro AUC: {rep['labels']['macro_auc']:.3f}")
    shutil.rmtree(out, ignore_errors=True)

studies: 4407 | gold: 58 | sites: 59 | eval labels: /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv

===== fold 0 | train 3525 val 882 =====
[f0] ep 1/1 loss 0.8014 | val macro-AUC 0.6004 | gold 0.4299 | 305s

CV macro-AUC (labels): 0.6003921652394438 | gold: 0.42990620490620496 | worst site: 0.5217026958115206

SMOKE OK in 5.9 min | fold-0 studies scored: 882 | 1-epoch macro AUC: 0.600


In [5]:
# ---- 5. long jobs: all arms, one queue per GPU (about 1.3 h per job; v5c waits for v5b) ----
results, t_start = {}, time.time()
def run_queue(gpu, names):
    for name in names:
        out = f"{W}/exp_{name}"
        if os.path.exists(f"{out}/oof.csv"):
            results[name] = "skipped (oof.csv already exists)"; continue
        with open(f"{W}/log_{name}.txt", "w") as lf:
            p = subprocess.Popen(train_cmd(name, out, ARMS[name]), env=dict(env, CUDA_VISIBLE_DEVICES=str(gpu)),
                                 stdout=lf, stderr=subprocess.STDOUT, cwd=CODE)
            results[name] = p.wait()

threads = [threading.Thread(target=run_queue, args=(g, q)) for g, q in QUEUES.items()]
for t in threads: t.start()
while any(t.is_alive() for t in threads):
    time.sleep(900)
    tails = {n: (open(f"{W}/log_{n}.txt").read().strip().splitlines() or [""])[-1][:110] for q in QUEUES.values() for n in q if os.path.exists(f"{W}/log_{n}.txt")}
    print(f"[{(time.time()-t_start)/3600:.2f} h]", tails)
for t in threads: t.join()
for n, rc in results.items():
    print(f"exp_{n} exit code {rc}")

[0.25 h] {'v5b': '[f0] ep 2/12 loss 0.7818 | val macro-AUC 0.7141 | gold 0.6594 | 374s', 'v4mix': '[f0] ep 2/12 loss 0.7792 | val macro-AUC 0.7147 | gold 0.6503 | 376s'}
[0.50 h] {'v5b': '[f0] ep 4/12 loss 0.7668 | val macro-AUC 0.7378 | gold 0.7284 | 379s', 'v4mix': '[f0] ep 4/12 loss 0.7616 | val macro-AUC 0.7412 | gold 0.7282 | 378s'}
[0.75 h] {'v5b': '[f0] ep 7/12 loss 0.7472 | val macro-AUC 0.7676 | gold 0.7745 | 376s', 'v4mix': '[f0] ep 7/12 loss 0.7418 | val macro-AUC 0.7674 | gold 0.7705 | 379s'}
[1.00 h] {'v5b': '[f0] ep 9/12 loss 0.7404 | val macro-AUC 0.7739 | gold 0.7554 | 374s', 'v4mix': '[f0] ep 9/12 loss 0.7349 | val macro-AUC 0.7731 | gold 0.7718 | 377s'}
[1.25 h] {'v5b': '[f0] ep 11/12 loss 0.7376 | val macro-AUC 0.7746 | gold 0.7524 | 379s', 'v4mix': '[f0] ep 11/12 loss 0.7316 | val macro-AUC 0.7753 | gold 0.7729 | 379s'}
[1.50 h] {'v5b': 'CV macro-AUC (labels): 0.77459059573173 | gold: 0.7524170274170274 | worst site: 0.6644546362728181', 'v5c': '[f0] ep 2/12 loss 0.

In [6]:
# ---- 6. analysis: all arms against the fixed old labels, with the noise floor from the baseline/seed runs ----
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
KEY4 = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"]
runs = {"baseline": BASE}
runs.update({k: v for k, v in SEED_OOF.items() if v})
runs.update({n: f"{W}/exp_{n}/oof.csv" for q in QUEUES.values() for n in q if os.path.exists(f"{W}/exp_{n}/oof.csv")})
D = {k: pd.read_csv(p).drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID") for k, p in runs.items()}
ref = next(k for k in D if k not in ("baseline", "seed7", "seed11"))
f0 = D[ref].index[D[ref]["p_ACL"].notna()]
for k in D: f0 = f0.intersection(D[k].index[D[k]["p_ACL"].notna()])
Y = D["baseline"].loc[f0, [f"y_{l}" for l in LABELS]].values
print("fold-0 studies:", len(f0), "| gold studies in fold 0:", int(D["baseline"].loc[f0, "is_gold"].sum()), "(too few for a gold comparison)")
P = {k: D[k].loc[f0, [f"p_{l}" for l in LABELS]].values for k in D}

def aucs(p, y=Y, idx=None):
    out = []
    for i in range(len(LABELS)):
        yi, pi = (y[:, i], p[:, i]) if idx is None else (y[idx, i], p[idx, i])
        m = (yi >= 0.9) | (yi <= 0.1)
        out.append(roc_auc_score(yi[m] >= 0.9, pi[m]) if len(np.unique(yi[m] >= 0.9)) == 2 else np.nan)
    return np.array(out)
k4 = [LABELS.index(l) for l in KEY4]
def table(pm):
    a = aucs(pm); return pd.Series(np.r_[a, np.nanmean(a), a[k4].mean()], index=LABELS + ["MACRO (12)", "KEY-4 mean"])

T = pd.DataFrame({k: table(P[k]) for k in P}).round(3)
print(T.to_string())
print("\nbaseline check: macro %.4f (expect ~0.779), KEY-4 %.4f (expect ~0.755)" % (T.loc["MACRO (12)", "baseline"], T.loc["KEY-4 mean", "baseline"]))
old_runs = [k for k in ("baseline", "seed7", "seed11") if k in T]
floor = T.loc["KEY-4 mean", old_runs].max() - T.loc["KEY-4 mean", old_runs].min() if len(old_runs) > 1 else 0.005
print(f"noise floor (KEY-4 range over {old_runs}): {floor:.4f}")

rng = np.random.default_rng(0); n = len(f0); B = 300
new_runs = [k for k in T if k not in old_runs]
for k in new_runs:
    d = T.loc["KEY-4 mean", k] - T.loc["KEY-4 mean", "baseline"]
    dm = T.loc["MACRO (12)", k] - T.loc["MACRO (12)", "baseline"]
    bs = [aucs(P[k], idx=(ix := rng.integers(0, n, n)))[k4].mean() - aucs(P["baseline"], idx=ix)[k4].mean() for _ in range(B)]
    lo, hi = np.nanpercentile(bs, [5, 95])
    verdict = "HARM: stop" if d < -0.01 else ("within noise" if abs(d) <= 0.01 else "possible gain (check on gold over 5 folds / public LB)")
    print(f"{k:7s} KEY-4 {d:+.4f} (90% CI [{lo:+.3f}, {hi:+.3f}])  macro {dm:+.4f}  -> {verdict}")
print("NB: scoring against the OLD labels favours models trained on the old labels, so a small drop is not proof the new labels are worse.")

rank = lambda p: np.column_stack([rankdata(c) / len(c) for c in p.T])
if len(old_runs) == 3:
    ens3 = sum(rank(P[k]) for k in old_runs) / 3
    print("\nrank-mean ensemble KEY-4: old 3 models %.4f" % aucs(ens3)[k4].mean())
    for k in new_runs:
        print("   + %-7s -> %.4f   (new-label model only: %.4f)" % (k, aucs((ens3 * 3 + rank(P[k])) / 4)[k4].mean(), T.loc["KEY-4 mean", k]))

fold-0 studies: 882 | gold studies in fold 0: 13 (too few for a gold comparison)
                  baseline  seed7  seed11    v5b    v5c  v4mix
ACL                  0.833  0.840   0.847  0.834  0.847  0.850
MCL                  0.713  0.730   0.717  0.710  0.716  0.729
Medial Meniscus      0.757  0.756   0.763  0.772  0.769  0.758
Lateral Meniscus     0.715  0.713   0.710  0.715  0.715  0.717
Medial OA            0.861  0.861   0.866  0.857  0.857  0.856
Lateral OA           0.749  0.736   0.746  0.738  0.733  0.729
PF OA                0.722  0.731   0.728  0.715  0.723  0.720
Effusion             0.801  0.797   0.812  0.796  0.795  0.791
Synovitis            0.799  0.800   0.795  0.784  0.785  0.784
Baker's              0.747  0.730   0.751  0.757  0.757  0.757
Contusion            0.808  0.801   0.807  0.801  0.798  0.789
Fracture             0.848  0.857   0.853  0.827  0.831  0.836
MACRO (12)           0.779  0.779   0.783  0.775  0.777  0.776
KEY-4 mean           0.755  0.760   0

**After it finishes:** commit (Save & Run All) so `exp_*` outputs become inputs. Decision rule: if an arm is within ~0.01 KEY-4 of the baseline on the old labels,
it is a candidate for the 5-fold retrain, where gold-58 over all folds and the public leaderboard can judge it. If it is clearly below, drop it.